<img src="../../images/SnowparkIconLabel.png" alt="Snowpark Icon" width=150px align=right /> 

# User-Defined Functions (UDFs) and User-Defined Table Functions (UDTFs)

Snowflake allows for UDFs to be developed in Scala, Java, or Python. This integrates nicely with Snowpark. You can create client-side objects and functions that will be serialized to your Snowflake virtual warehouse for parallel execution much like standard, built-in functions and UDFs written in JavaScript or SQL. 

Here we will only discuss Python UDFs. Python UDFs are not limited to being used solely with Snowpark code. You can even register your Python UDFs so they can be invoked in SQL statements. These UDFs can be temporary (life of the session) or permanent (until explicitly dropped).

Additionally, Python UDFs can be written to read files from stages. 

This module explores Snowpark Python User-Defined Functions (UDFs) and Python User-Defined Table Functions (UDTFs, a.k.a. "tabular functions").

> **&#128221; Note:** UDFs and UDTFs are allowed to read and write to the `/tmp` directory of the nodes in the virtual warehouse that is executing them. However, there are limits. Snowflake users should be conservative with these limits and know that these limits can change without warning.  For standard warehouses, the limit a UDF can write to `/tmp` is `500MB`. For a Snowpark Optimized Warehouse, the limit is `8GB`. These are "soft" limits. Snowflake users may consult with their account rep to have these limits adjusted for their particular needs.


**Goals:** Create and register UDFs and UDTFs, add a dependency and read from files within a UDF handler function

## Topics in this Lesson

1. [Snowpark In-Line Client-side Created UDFs](#Snowpark_In_Line_Client_side_Created_UDFs)  
    1. [Pickling](#Pickled)  
    1. [Lambda User-Defined Functions](#Lambda_User_Defined_Functions)  
    1. [Named UDFs](#Named_UDFs)    
1. [Registering a UDF](#Registering_a_UDF)   
    1. [Temporary UDFs](#Temporary_UDFs)  
    1. [Permanent UDFs](#Permanent_UDFs)  
1. [Adding Dependencies](#Adding_Dependencies)  
1. [Python Library UDFs](#Python_Library_UDFs)  
1. [Creating and Using Python User-Defined Table Functions (UDTFs)](#Creating_and_Using_Python_User-Defined_Table_Functions_UDTFs)  
    1. [A Brief Review of Snowflake SQL Table Functions](#A_Brief_Review_of_Snowflake_SQL_Table_Functions)  
    1. [And This Pertains to Snowpark How?](#And_This_Pertains_to_Snowpark_How)  
    1. [The Steps to Developing a Python UDTF](#The_Steps_to_Developing_a_Python_UDTF)  
    1. [It's Just a Class - It's Just Methods](#It_s_Just_a_Class_It_s_Just_Methods)  
    1. [Registering a Python UDTF](#Registering_a_Python_UDTF)  
    1. [Invoking Python UDTFs](#Invoking_Python_UDTFs)  
    1. [Using a Python UDTF in SQL](#Using_a_Python_UDTF_in_SQL)  
1. [Clean Up and Close Session](#AE6_cleanup)

 

---

<a id="Snowpark_In_Line_Client_side_Created_UDFs"></a>
## 1. Creating and Registering UDFs

Snowpark has `Session` functionality for creating UDFs as well as a class named `UserDefinedFunction`. Within your Python application, you can create classes and methods, in line, that can be used temporarily or permanently as Snowpark UDFs as well as Snowflake SQL UDFs. 

- Serialize and register code as a Snowflake UDF
- Execute within Snowflake Virtual Warehouse
    - Containerized
    - Elastic
    - Secure

<img src="../../images/python_objects.png" alt="PythonObjects" style="width:90%;display:block;margin-left:3%;" />
<br />


> **&#128221; Note:** See documentation for further details:
> - [Snowflake docs: Creating User-Defined Functions (UDFs) for DataFrames in Python](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-udfs.html#creating-user-defined-functions-udfs-for-dataframes-in-python)
> - [snowflake.snowpark.udf.UserDefinedFunction(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.udf.UserDefinedFunction.html#snowflake-snowpark-udf-userdefinedfunction)
> - [snowflake.snowpark.Session.udf](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.udf.html#snowflake-snowpark-session-udf)
> - [snowflake.snowpark.udf.UDFRegistration(Session)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.udf.UDFRegistration.html#snowflake-snowpark-udf-udfregistration)
> - [snowflake.snowpark.udtf.UserDefinedTableFunction(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.udtf.UserDefinedTableFunction.html#snowflake-snowpark-udtf-userdefinedtablefunction)
> - [snowflake.snowpark.Session.udtf](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.udtf.html#snowflake-snowpark-session-udtf)
> - [snowflake.snowpark.udtf.UDTFRegistration(Session)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.udtf.UDTFRegistration.html#snowflake.snowpark.udtf.UDTFRegistration)

#### Connect and create a `Session` 
1. Import required libraries.  
1. Create a `Session` to connect to Snowflake.  
<br />  
    > &#10071; Success requires that you have already completed the key pair authentication exercise.  
1. Set context items for this module.  

Run the following cell to connect to your Snowflake account. *You needn't edit anything in the following cell. Just run it.*

In [1]:
# Run utils notebook
%run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session
session = create_session()

Read of key/pair authentication objects successful
User authenticated and session created
----------------------------------------------------------
|"Session Property Name"      |"Session Property Value"  |
----------------------------------------------------------
|version                      |1.6.1                     |
|python.version               |3.8.18                    |
|python.connector.version     |3.7.0                     |
|python.connector.session.id  |396816324034406           |
|os.name                      |Linux                     |
----------------------------------------------------------



---

#### Setup Code

We should set ourselves up for success. The following ensures our context is set properly for database, schema, role, and warehouse. *You needn't edit anything in the following cell. Just run it.*  Give it a few seconds to complete. When finished, you should see the message:
```
========== READY ===========
```

In [2]:
# Hard code the lesson name
lesson_name = "UDFS_AND_UDTFS_PY"

# Create the context items for this lesson
lesson = confirm_or_create_lesson_context(session, lesson_name)

Creation of context items could take a few moments... be patient
The current user is: RAVEN
Query tag is: Data Science: RAVEN - UDFS_AND_UDTFS_PY
-------------------------------------------------
|"Created warehouse RAVEN_WH"                   |
-------------------------------------------------
|RAVEN_WH already exists, statement succeeded.  |
-------------------------------------------------

------------------------------------
|"Altered warehouse RAVEN_WH"      |
------------------------------------
|Statement executed successfully.  |
------------------------------------

Setting current warehouse to RAVEN_WH
Creating database RAVEN_DB
-------------------------------------------------
|"Created database RAVEN_DB"                    |
-------------------------------------------------
|RAVEN_DB already exists, statement succeeded.  |
-------------------------------------------------

Creating schema UDFS_AND_UDTFS_PY_LESSON
---------------------------------------------------------
|"

<a id="Pickled"></a>
### 1A. Pickling

Creating UDFs in client code means the UDF will be pickled from the client app to your Snowflake account for execution within your Virtual Warehouse. 

<img src="../../images/RegisterUDFPython.png" alt="RegisterUDF" style="width:85%;display:block;margin-left:5%;" />
<br />

> **&#128221; Note:** See documentation for further details:
> - docs.python.org [Python Object Serialization - `pickle`](https://docs.python.org/3.8/library/pickle.html#module-pickle)
> - TutorialsPoint [Python Pickling](https://www.tutorialspoint.com/python-pickling)

<a id="Lambda_User_Defined_Functions"></a>
### 1B. Python Lambda UDFs

For "one-off" logic, you might want to consider a Python Lambda UDF. There is no name for the function. It's not reusable. Sounds bad but you will see them a lot. 

A Python Lambda UDF can be passed to the built-in function `snowflake.snowpark.functions.udf(...)` to create an instance of `snowflake.snowpark.udf.UserDefinedFunction`. This object can then be used in transformations of `DataFrame` objects.

<a id="udf"></a>

#### `functions.udf(...)`

The function `Snowflake.snowpark.functions.udf(...)` takes the following arguments:
- **func** - The Python function object. Can be a lambda function or a named function.
- **return_type** - A `DataType` object representing the return data type of the UDF (optional)
- **input_types** - A list of `DataType` objects representing the input data types of the UDF (optional)
- **name** - A string or list of strings that specify the name or fully-qualified object identifier (database name, schema name, and function name) (optional). If it is not provided, a name will be automatically generated for the UDF. A name is mandatory when `is_permanent` is `True`.
- **is_permanent** – Will the UDF be permanent or temporary. The default is `False`. If it is `True`, a valid `stage_location` must be provided.
- **stage_location** – The stage where the Python file for the UDF and its dependencies should be uploaded (optional) Mandatory when `is_permanent` is `True`. Ignored when `is_permanent` is `False`. Cannot be a temporary stage or an external stage. 
- **imports** -  A list of imports for this UDF. Can be a path string or a tuple of two strings to represent a file path and an import path (similar to the `import_path` argument in `add_import()`) (optional)
- **packages** – A list of packages for this UDF. These UDF-level packages will override the session-level packages added by `add_packages()` and `add_requirements()` (optional).
- **replace** – Overwrite an existing UDF with the same name? The default is `False`. If it is `False`, attempting to register a UDF with a name that already exists results in a `SnowparkSQLException` being raised. If set to `True`, an existing UDF with the same name is overwritten.
- **session** – Use this session to register the UDF. If it’s not specified, the session that you created before calling this function will be used. You need to specify this parameter if you have created multiple sessions before calling this method.

See the [docs](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.udf.html#snowflake.snowpark.functions.udf) for other parameters (`parallel`, `max_batch_size`, and `statement_params`). 

```python
from snowflake.snowpark.functions import udf

my_udf = (functions.udf(
        <lambda function here> 
    )
)         
    
(my_df.with_column(
     "COL_NAME"
    ,my_udf(
         <arg column 1>
        ,<arg column 2>
        )
    )
   .show()
)      
```

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.functions.udf(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.udf.html#snowflake.snowpark.functions.udf)
> - [snowflake.snowpark.udf.UserDefinedFunction(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.udf.UserDefinedFunction.html#snowflake.snowpark.udf.UserDefinedFunction)

In [3]:
# The function is nested within a Snowpark UDF object 
from snowflake.snowpark.types import DoubleType
import math

 # Create the UDF object
my_lambda_circum_udf = (functions   
    .udf(                             # func is the lambda function
         func = lambda radius: 2.0 * math.pi * radius
        ,return_type = DoubleType()   # Our lambda's returns type
        ,input_types = [DoubleType()] # List of argument types
        ,is_permanent = False         # Temp or permanent UDF? 
#        ,stage_location = '@~'       # Ignored when is_permanent is False
        ,replace = True               # Overwrite existing UDF with the same name if exists
        ,session = session
     )
            
)  
print(f"The name of the function is {my_lambda_circum_udf.name}")

The name of the function is "RAVEN_DB"."UDFS_AND_UDTFS_PY_LESSON".SNOWPARK_TEMP_FUNCTION_106ZKGOHZK


--- 

Notice above that the function has been automatically and dynamically named. We can provide our own name if we pass the `name` argument like below. The name can be a single string or a list of strings holding the database name, schema name, and function name.

In [4]:
# Retrieve the current database and schema from the Session object
curr_db = session.get_current_database()
curr_schema = session.get_current_schema()

from snowflake.snowpark.types import DoubleType
import math

# Same as above, but we used the "name" argument
my_lambda_circum_udf = (functions
    .udf(                           
         func = lambda radius: 2.0 * math.pi * radius
        ,return_type = DoubleType()
        ,input_types = [DoubleType()]
        ,is_permanent = False       
        ,stage_location = '@~'      
        ,replace = True             
        ,session = session
        ,name = [curr_db, curr_schema, "LAMBDA_CIRCUMFERENCE"] # DB, Schema, and SQL function name
     )            
) 
print(f"The name of the function is {my_lambda_circum_udf.name}")

# Print the class name of our UDF object
print(f"The class of my_lambda_circum_udf is: {type(my_lambda_circum_udf)}")

The name of the function is "RAVEN_DB"."UDFS_AND_UDTFS_PY_LESSON".LAMBDA_CIRCUMFERENCE
The class of my_lambda_circum_udf is: <class 'snowflake.snowpark.udf.UserDefinedFunction'>


---
Let's test our UDF/lambda function programmatically. 

In [5]:
# We test it on simple data and display our results
from snowflake.snowpark.functions import col
(session.create_dataframe([1,2,3,4,5])
    .to_df("RADIUS")
    .with_column(
             "THE_CIRCUM"
            ,my_lambda_circum_udf("RADIUS")
     )
    .sort(col("RADIUS").asc())
    .show() 
) 

---------------------------------
|"RADIUS"  |"THE_CIRCUM"        |
---------------------------------
|1         |6.283185307179586   |
|2         |12.566370614359172  |
|3         |18.84955592153876   |
|4         |25.132741228718345  |
|5         |31.41592653589793   |
---------------------------------



<a id="Named_UDFs"></a>
### 1C. Named UDFs

Creating a function object with a name allows it to be used multiple times and in other contexts. Using [decorators](https://python-3-patterns-idioms-test.readthedocs.io/en/latest/PythonDecorators.html) by annotating the function provides the necessary context for it to be a Snowpark UDF. 

```python
@udf(name = "<sql_function_name>", input_types = [...], return_type = <XXXType()>, ...) 
def myFunction(<args list here>) :
    <some return value>
```
By specifying a `name` argument to the `@udf(...)` annotation, we can then use the UDF in SQL statements like the below. 

```sql
SELECT <sql_function_name>(<arg>)...;
```


In [6]:
import math
from snowflake.snowpark.types import DoubleType
from snowflake.snowpark.functions import udf, col

def circumference_func(radius) -> float:
    return 2.0 * radius * math.pi 

# Creating a UDF object by invoking functions.udf(...) and passing in the powering function plus more
my_udf_object = (functions
    .udf(  # Returns a UserDefinedFunction object
         name = "my_area_udf"         # Name to be used in SQL select statements
        ,func = circumference_func    # <--- THE PYTHON FUNCTION POWERING THIS UDF
        ,return_type = DoubleType()   # The return type of the UDF
        ,input_types = [DoubleType()] # The data types for the argument(s) of the UDF
        ,is_permanent = False         # Will this UDF live past this Snowpark session? 
        ,replace = True               # If a function with this name and same args exists? overwrite it?
        ,session = session            # The Snowpark session object (IOW, on which SF account will this UDF be used?)
    )    
)

# The above shows using functions.udf(...) but we don't actually use the created myUDFobject.
# The above was meant to be a syntax example that you can compare/contrast with the code below that uses the decorator pattern.

@udf(name = "my_circum_udf"
    ,return_type = DoubleType()
    ,input_types = [DoubleType()]
    ,is_permanent = False
    ,replace = True
    ,session = session)
def circumference(radius) -> float:
    return 2.0 * radius * math.pi 

@udf(name = "my_area_udf"
     ,return_type = DoubleType()
     ,input_types = [DoubleType()]
     ,is_permanent = False
     ,replace = True
     ,session = session)
def circle_area(radius) -> float:
    return math.pi * math.pow(radius,2)

print(f"Class of circumference is {type(circumference)}")
print(f"Class of circle_area is {type(circle_area)}")

# Use them in transformations
(session.create_dataframe([1,2,3,4,5])
    .to_df("RADIUS")
    .with_column("CIRCUMFERENCE", circumference(col("RADIUS")))
    .with_column("AREA", circle_area(col("RADIUS")))
    .sort(col("RADIUS").asc())
    .show()
)

Class of circumference is <class 'snowflake.snowpark.udf.UserDefinedFunction'>
Class of circle_area is <class 'snowflake.snowpark.udf.UserDefinedFunction'>
------------------------------------------------------
|"RADIUS"  |"CIRCUMFERENCE"     |"AREA"              |
------------------------------------------------------
|1         |6.283185307179586   |3.141592653589793   |
|2         |12.566370614359172  |12.566370614359172  |
|3         |18.84955592153876   |28.274333882308138  |
|4         |25.132741228718345  |50.26548245743669   |
|5         |31.41592653589793   |78.53981633974483   |
------------------------------------------------------



In [7]:
# The name argument to the @udf decorator allows for use in SQL text
(session.sql("SELECT 2 AS RADIUS, my_circum_udf(RADIUS), my_area_udf(RADIUS)")
     .show()
)

--------------------------------------------------------------
|"RADIUS"  |"MY_CIRCUM_UDF(RADIUS)"  |"MY_AREA_UDF(RADIUS)"  |
--------------------------------------------------------------
|2         |12.566370614359172       |12.566370614359172     |
--------------------------------------------------------------



---

<a id="Registering_a_UDF"></a>
## 2. Registering a UDF

A UDF in Snowpark can be registered as temporary or permanent. We use the `snowflake.snowpark.Session` instance `session` to register the UDF. Session contains an instance of `UDFRegistration` which has a `.register(...)` function that returns a `UserDefinedFunction` object. 

The function `UDFRegistration.register(...)` takes the same arguments as `functions.udf(...)` [listed above](#udf) and repeated below:
- **func** - The Python function object. Can be a lambda function or a named function.
- **return_type** - A `DataType` object representing the return data type of the UDF (optional)
- **input_types** - A list of `DataType` objects representing the input data types of the UDF (optional)
- **name** - A string or list of strings that specify the name or fully-qualified object identifier (database name, schema name, and function name) (optional). If it is not provided, a name will be automatically generated for the UDF. A name is mandatory when `is_permanent` is `True`.
- **is_permanent** – Will the UDF be permanent or temporary. The default is `False`. If it is `True`, a valid `stage_location` must be provided.
- **stage_location** – The stage where the Python file for the UDF and its dependencies should be uploaded (optional) Mandatory when `is_permanent` is `True`. Ignored when `is_permanent` is `False`. Cannot be a temporary stage or an external stage. 
- **imports** -  A list of imports for this UDF. Can be a path string or a tuple of two strings to represent a file path and an import path (similar to the `import_path` argument in `add_import()`) (optional)
- **packages** – A list of packages for this UDF. These UDF-level packages will override the session-level packages added by `add_packages()` and `add_requirements()` (optional).
- **replace** – Overwrite an existing UDF with the same name? The default is `False`. If it is `False`, attempting to register a UDF with a name that already exists results in a `SnowparkSQLException` being raised. If set to `True`, an existing UDF with the same name is overwritten.
- **session** – Use this session to register the UDF. If it’s not specified, the session that you created before calling this function will be used. You need to specify this parameter if you have created multiple sessions before calling this method.

See the [docs](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/_autosummary/snowflake.snowpark.functions.html#snowflake.snowpark.functions.udf) for lesser-used parameters (`parallel`, `max_batch_size`, and `statement_params`). 


> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Session.udf.register(...)](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-udfs#creating-and-registering-a-named-udf)
> - [snowflake.snowpark.udf.UDFRegistration(Session)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.udf.UDFRegistration.html#snowflake-snowpark-udf-udfregistration)

<a id="Temporary_UDFs"></a>
### 2A. Temporary UDFs
A temporary UDF will be deleted once the Snowpark/Snowflake session ends. To use it again later, you'll need to re-register it.

```python
my_udf = (session.udf
    .register(
         func = <a UDF object>
        ,name = "<function name>"
        ,is_permanent = False  # False means temporary UDF, no stage_location is required
        ,...
      )
    )
```



In [8]:
# Create a UDF object and register it as temporary
from snowflake.snowpark.types import DoubleType

# Creating the same Python function that we created before hurts nothing.
# Repeating this code is for clarity. 
def circumference(radius) -> float:
    return 2.0 * radius * math.pi 

# Notice the argument is_permanent
circum_udf_temp = (session
    .udf                      # An instance of UDFRegistration
    .register(
         func = circumference
        ,name = "CALCULATE_CIRCUMFERENCE_TEMP"
        ,return_type = DoubleType()
        ,input_types = [DoubleType()]
        ,is_permanent = False # The UDF will be temporary
        ,replace = True       # Avoid errors in case we run this cell more than once
        ,session = session    # Use the session for this lesson for registration
    )
) 

---
The UDF is created and registered. We can use it programmatically or in SQL text (below). 

In [9]:
# Confirm it
from snowflake.snowpark.functions import col, split, lit
(session.sql("SHOW USER FUNCTIONS LIKE 'CALCULATE_CIRCUMFERENCE_TEMP'")
    .select(
         col('"name"')
        ,split(split(col('"arguments"'),lit("("))[1],lit(")"))[0].alias("ARGS")
        ,col('"language"')
        ,col('"is_builtin"')
      )
    .show(1,60)
)

----------------------------------------------------------------------
|"name"                        |"ARGS"   |"language"  |"is_builtin"  |
----------------------------------------------------------------------
|CALCULATE_CIRCUMFERENCE_TEMP  |"FLOAT"  |PYTHON      |N             |
----------------------------------------------------------------------



---

We can use this temporary function for the duration of our session.

```python
# Programmatically
(my_df.with_column(
     "<col name>"
    ,circumUDFTemp(<some numeric column>)
  )
 .show()
)
```
or
```python
# Via SQL
session.sql("SELECT CALCULATE_CIRCUMFERENCE_TEMP(<some number>)")
```

In [10]:
# Hard-code an example
num = 10

# Test our UDF programmatically
print("""Invoking our UDF programmatically:\n...with_column("CIRCUM",circum_udf_temp(col("NUM"))""")
from snowflake.snowpark.functions import lit, col
(session.create_dataframe([""])
    .select(
         lit(num).alias("NUM")  
     )
    .with_column(
         "CIRCUM"
        ,circum_udf_temp(col("NUM"))
     )
    .show()
) 

# Test our UDF via SQL
select_text = f"SELECT {num} AS NUM, CALCULATE_CIRCUMFERENCE_TEMP({num}) AS CIRCUM"
print(f"Invoking our UDF via SQL:\n{select_text}")
(session.sql(select_text)
    .show()
)    

Invoking our UDF programmatically:
...with_column("CIRCUM",circum_udf_temp(col("NUM"))
-----------------------------
|"NUM"  |"CIRCUM"           |
-----------------------------
|10     |62.83185307179586  |
-----------------------------

Invoking our UDF via SQL:
SELECT 10 AS NUM, CALCULATE_CIRCUMFERENCE_TEMP(10) AS CIRCUM
-----------------------------
|"NUM"  |"CIRCUM"           |
-----------------------------
|10     |62.83185307179586  |
-----------------------------



<a id="Permanent_UDFs"></a>
### 2B. Permanent UDFs
A permanent UDF can be invoked in other sessions and by other users in other roles. It's an object, in a schema, that can be invoked within SQL statements like other UDFs you might've created using JavaScript, SQL, Java, Scala, or Python. 

Because this is a Python UDF, the handler class will need to be placed into a stage.

```python
(session.sql("CREATE STAGE IF NOT EXISTS <stage name>")
    .show()
)

(session.udf.register(
         <args similar to temporary function above>
        ,is_permanent = True                 # Makes this UDF permanent
        ,stage_location = "@<some stage>"    # Required when is_permanent = True
        ,<args similar to temporary function above>
    )
) 
```

> &#10071; ***A permanent UDF will require a stage*** in which Snowpark can place the `.py` file that contains your UDF Python code. If the pickled hex code of the Python function is quite small (8,192 pickled hex characters or less), the function will be stored as metadata about the UDF and no .py file will be placed into the stage - but the stage must still be specified with `stage_location`. 

> &#10071; ***Known bug:*** If one supplies a value for `stage_location` the value of `is_permanent` will assumed to be `True` when it is not specified and even when it is indeed specified and set to `False`. For forward compatibility, one should **not** use this bug to avoid setting `is_permanent`. Set the value to `True` when you want a permanent UDF in case you rebuild and replace your UDF at a later date with a later version of Snowpark.

In [11]:
# Come up with a name for our stage
full_stage_name = f"{lesson.full_schema()}.MY_UDF_STAGE"

# Create a stage in which we place our permanent UDF
print("Creating the stage if it doesn't already exist.")
(session
    .sql(f"CREATE STAGE IF NOT EXISTS {full_stage_name}")
    .show(1,60)
)

# Print the name
print(f"Fully-qualified stage name is {full_stage_name}")

# Create the UDF and register it as permanent
from snowflake.snowpark.types import DoubleType

# The simple Python function (again)
def circumference(radius) -> float:
    return 2.0 * radius * math.pi 

# Create the UserDefinedFunction instance
print("Creating the UDF object with the SQL name 'CALCULATE_CIRCUMFERENCE'")
circum_udf = (session.udf
    .register(
         func = circumference              # The Python function
        ,name = "CALCULATE_CIRCUMFERENCE"  # Name used when using the UDF in SQL text
        ,return_type = DoubleType()
        ,input_types = [DoubleType()]
        ,is_permanent = True                    # This UDF can be used after this session ends
        ,stage_location = f"@{full_stage_name}" # Stage location is mandatory when is_permanent is True
        ,replace = True
        ,session = session   
    )
)

print(f"UDF created with name {circum_udf.name}")

Creating the stage if it doesn't already exist.
-------------------------------------------------
|"status"                                       |
-------------------------------------------------
|Stage area MY_UDF_STAGE successfully created.  |
-------------------------------------------------

Fully-qualified stage name is RAVEN_DB.UDFS_AND_UDTFS_PY_LESSON.MY_UDF_STAGE
Creating the UDF object with the SQL name 'CALCULATE_CIRCUMFERENCE'
UDF created with name CALCULATE_CIRCUMFERENCE


---

We can confirm that the function exists with a simple `SHOW USER FUNCTIONS` statement.

In [12]:
# Verify the function exists
print("Showing user functions like '%CIRCUM%'")
(session.sql("SHOW USER FUNCTIONS LIKE '%CIRCUM%'")
    .select(
         col('"name"')
                    # extract args list from "function_name(arg1,arg2)"
        ,split(split(col('"arguments"'),lit("("))[1],lit(")"))[0].alias("ARGS")
        ,col('"language"')
        ,col('"is_builtin"')
      )
    .show(1,60)
)

Showing user functions like '%CIRCUM%'
-----------------------------------------------------------------
|"name"                   |"ARGS"   |"language"  |"is_builtin"  |
-----------------------------------------------------------------
|CALCULATE_CIRCUMFERENCE  |"FLOAT"  |PYTHON      |N             |
-----------------------------------------------------------------



---

Usually, we can confirm that a `.py` file was created and placed into our stage using `LIST`.  However, when the pickled hex code is short (8,192 characters or less in hex code), Snowflake will directly inline the code in the function definition.  

In [13]:
# Look in the stage for the generated .py file
print(f"The content of our stage {full_stage_name} is:")
(session.sql(f"LIST @{full_stage_name}")
    .show()
)

The content of our stage RAVEN_DB.UDFS_AND_UDTFS_PY_LESSON.MY_UDF_STAGE is:
---------------------------------------------
|"name"  |"size"  |"md5"  |"last_modified"  |
---------------------------------------------
|        |        |       |                 |
---------------------------------------------



---

<span style="color:red;">Nothing showing above?</span> That's because our UDF is quite short.

Notice below that when we describe the function, the bytes for the body are hard-coded. For a more involved UDF, a file would be written to the specified stage. 

In [14]:
# Describe the user function
from snowflake.snowpark.functions import col

# Extract the function body from the function declaration
function_body = (session.sql(f"DESCRIBE FUNCTION CALCULATE_CIRCUMFERENCE(DOUBLE)")
     .filter(col('"property"') == lit("body"))
     .select('"value"')
     .collect()[0][0]
)

# Print the body
print("="*20,"Begin Function Body","="*20)
print(function_body)
print("="*20,"End Function Body","="*20)

==================== Begin Function Body ====================

import pickle

func = pickle.loads(bytes.fromhex('80059505020000000000008c17636c6f75647069636b6c652e636c6f75647069636b6c65948c0d5f6275696c74696e5f747970659493948c0a4c616d6264615479706594859452942868028c08436f6465547970659485945294284b014b004b004b014b024b43430e64017c00140074006a0114005300944e47400000000000000086948c046d617468948c0270699486948c067261646975739485948c212f746d702f6970796b65726e656c5f313537392f313332373939373437342e7079948c0d63697263756d666572656e6365944b1243020001942929749452947d94288c0b5f5f7061636b6167655f5f944e8c085f5f6e616d655f5f948c085f5f6d61696e5f5f94754e4e4e749452948c1c636c6f75647069636b6c652e636c6f75647069636b6c655f66617374948c125f66756e6374696f6e5f7365747374617465949394681a7d947d9428681768118c0c5f5f7175616c6e616d655f5f9468118c0f5f5f616e6e6f746174696f6e735f5f947d948c0e5f5f6b7764656661756c74735f5f944e8c0c5f5f64656661756c74735f5f944e8c0a5f5f6d6f64756c655f5f9468188c075f5f646f635f5f944e8c0b5f5f636c6f737572655

---
The describe above is a SQL DESCRIBE FUNCTION. We also have `DataFrame.describe(...)` which produces different results.

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.DataFrame.describe(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.DataFrame.describe.html#snowflake.snowpark.DataFrame.describe)

---

Just as with a registered temporary UDF, we can use our permanent UDF in both `DataFrame` transformations as well as SQL text. The difference is that we can continue to use our permanent UDF *after* this session ends. 

If you use a Snowflake worksheet to see the user functions, you will find it there even **after this session ends**. 

``` sql
SHOW USER FUNCTIONS LIKE '%CIRCUM%'; 
```

In [15]:
# Test our UDF programmatically
from snowflake.snowpark.functions import lit,col

# Test and show results programmatically
print("""Programmatically testing our function using\n....with_column("CIRCUM",circum_udf(col("NUM")))""")
(session.create_dataframe([1.0,1.5,2.0,9.0,10.0]).to_df("NUM")
    .with_column(
         "CIRCUM"
        ,circum_udf(col("NUM"))
     )
    .sort(col("NUM").asc())
    .show()
) 

# Test our UDF using SQL text
select_text = f"SELECT {num} AS NUM, CALCULATE_CIRCUMFERENCE({num}) AS CIRCUM"
print(f"Texting our function in SQL text: {select_text}")
(session.sql(select_text)
    .show()
) 

Programmatically testing our function using
....with_column("CIRCUM",circum_udf(col("NUM")))
------------------------------
|"NUM"  |"CIRCUM"            |
------------------------------
|1.0    |6.283185307179586   |
|1.5    |9.42477796076938    |
|2.0    |12.566370614359172  |
|9.0    |56.548667764616276  |
|10.0   |62.83185307179586   |
------------------------------

Texting our function in SQL text: SELECT 10 AS NUM, CALCULATE_CIRCUMFERENCE(10) AS CIRCUM
-----------------------------
|"NUM"  |"CIRCUM"           |
-----------------------------
|10     |62.83185307179586  |
-----------------------------



---

<a id="Adding_Dependencies"></a>
## 3. Adding Dependencies

If your UDF uses other resources such as libraries, you can register these files as "dependencies". 

#### Dependency Files in a Stage

The steps:
1. Create a stage if you don't already have one in mind
1. Put the file(s) into the stage
1. Use your `Session` instance to add the file(s) as a dependency


```python
# Create the stage
(session.sql("CREATE STAGE MYSTAGE")
    .show()
)

# Put files in the stage
# See Appendix on using snowflake.snowpark.FileOperation.put(...)

# Add the library file as an import
session.add_import("@my_stage/<path>/my_library.zip") 

# Add a directory as an import
session.add_import("/<path>/my-resource-dir/") 

# Add any type of file you might need
session.add_import("/<path>/my-resource.xml") 
```

#### Dependency Files on Your Local System

- Use your `Session` instance to add the file(s) as a dependency

```python
session.add_import("/<local>/<path>/my_module.py")  
```

> **&#128221; Note:** See documentation for further details:
> - [Specifying Dependencies for a UDF](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-udfs.html#specifying-dependencies-for-a-udf)
> - [snowflake.snowpark.Session.add_import(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.add_import.html#snowflake.snowpark.Session.add_import)



---

<a id="Python_Library_UDFs"></a>
## 4. Python Library UDFs

If your functionality already exists in a Python library, you can take the steps above to register the dependency and then reference the function in your Snowpark code no differently than you did when you wrote the function in-line. You are not required to develop the function within your Snowpark application.

Additionally, you can specify Anaconda packages to install when you create Python UDFs.  Use `Session.add_packages(...)` to add packages.

```python
session.add_packages("numpy", "pandas", "xgboost==1.5.0")
```

You can also use `session.add_requirements(...)` to specify packages with a requirements file. “Requirements files” are files containing a list of items to be installed using `pip install`.

```python
session.add_requirements("mydir/requirements.txt")  
```
> **&#128221; Note:** See documentation for further details:
> - [Requirements Files](https://pip.pypa.io/en/stable/user_guide/#requirements-files)
> - [snowflake.snowpark.Session.add_packages(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.add_packages.html#snowflake.snowpark.Session.add_packages)
> - [snowflake.snowpark.Session.add_requirements(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.add_requirements.html#snowflake.snowpark.Session.add_requirements)


---

<a id="Creating_and_Using_Python_User-Defined_Table_Functions_UDTFs"></a>

## 5. Creating and Using Python User-Defined Table Functions (UDTFs)


Along with built-in table functions, Snowpark supports user-defined table functions, a.k.a. UDTFs, a.k.a. "tabular functions". 

> **&#128221; Note:** See documentation for further details:
> - [Snowpark Developers Guide: Creating User-Defined Table Functions (UDTFs) for DataFrames in Python](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-udtfs.html#creating-user-defined-table-functions-udtfs-for-dataframes-in-python)

<a id="A_Brief_Review_of_Snowflake_SQL_Table_Functions"></a>
### 5A. A Brief Review of Snowflake SQL Table Functions

A "table function" is a function that potentially returns zero or more rows of data (barring an error). This is quite different from a "scalar function" that returns a single value (e.g. `AVG(...)`, `RANDOM()`, `SUBSTR(...)`). Because the results of a table function are rows, the invocation needs to be passed into the built-in Snowflake function `TABLE(...)`. This creates an anonymous, temporary table from which one can select columns.

```sql
SELECT * FROM TABLE(SOME_TABLE_FUNCTION(...)) WHERE ...;
```

> **&#128221; Note:** See documentation for further details:
> - [Snowflake Docs: Table Functions](https://docs.snowflake.com/en/sql-reference/functions-table.html#table-functions)


<a id="And_This_Pertains_to_Snowpark_How"></a>
### 5B. And This Pertains to Snowpark How? 

Snowpark supports calling built-in table functions through the class `snowflake.snowpark.table_function.TableFunctionCall`.

Your custom logic is a bit more complex with a UDTF over a scalar UDF. Because a UDTF produces potentially zero or more rows, you will need to define a schema for your results (a `StructType`) and provide certain lifecycle methods.

The constructor of `TableFunctionCall` is not supposed to be called directly. Instead, use `snowflake.snowpark.functions.call_table_function(...)` to create an instance of this class or `snowflake.snowpark.functions.table_function(...)`.  Note that table_function() returns a `Callable` and not a `TableFunctionCall` instance.

Additionally, one can use `snowflake.snowpark.Session.table_function(...)` which returns a `DataFrame`. 

<img src="../../images/function_return.png" alt="Function Return" width=65% /> 

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.table_function.TableFunctionCall(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.table_function.TableFunctionCall.html#snowflake-snowpark-table-function-tablefunctioncall)
> - [snowflake.snowpark.functions.call_table_function(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.call_table_function.html#snowflake.snowpark.functions.call_table_function)
> - [snowflake.snowpark.functions.table_function()](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.table_function.html#snowflake.snowpark.functions.table_function)
> - [snowflake.snowpark.Session.table_function(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.table_function.html#snowflake.snowpark.Session.table_function)

<a id="The_Steps_to_Developing_a_Python_UDTF"></a>
### 5C. The Steps to Developing a Python UDTF

1. Write a UDTF handler class. This class must implement methods that Snowflake invokes when the UDTF is called.
1. Implement the following lifecycle (a.k.a. callback) methods:
    - `__init__` (**optional**)
        - Declared as such: `def __init__(self):`
        - Called only once by the platform
        - Can take only `self` as an argument.
        - Invoked once per partition to initialize stateful processing of input partitions
        - May not produce output rows
    - `process` (<span style="color:red;">**required**</span>)
        - Declared as such: `def process(self, *args):`
        - Must have a `self` parameter
        - Invoked for each input row processed
        - UDF Design Optimizations (See: [Designing Python UDFs](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-designing.html#label-sql-python-data-type-mappings))
        - Returns a tabular value as tuples that correspond to the sequence of UDTF return value columns
        - The return could be an empty list
        - An exception will cause processing to stop
    - `end_partition` (**optional**)
        - Declared as such: `def end_partition(self):`
        - May not have any parameters other than `self`
        - Invoked to finalize processing of input partitions
        - Called once after all rows/input values are processed for each single partition of data
        - Returns either a tabular value, an empty list or `None`
1. Register the UDTF as temporary or permanent function for dual use in application code and SQL statements. (**optional**)      

```python 

class MyHandler:
    
    # Optional - called once per partition
    def __init__(self):
        self.some_value = <value>
        self.some_other_value = <value>
    
    # Required - called with each input row processed
    def process(self, arg1, arg2, arg3):
      # Use args, use self values, produce tuples of values
      # or produce no tuples 
      yield (col1,col2,col3,col4,...)

    # Optional - called once per partition
    def end_partition(self):
      # Produce tuples or no tuples
      yield (col1,col2,col3,col4,...)
        
# Register table function like any scalar function

from snowflake.snowpark.functions import udtf
from snowflake.snowpark.types import StructType, StructField, XXXType, YYYType, ZZZType
my_udtf_instance = (
    udtf(
         handler = MyHandler         # The class for the handler
        ,output_schema = StructType( # The schema of the results output
            [  StructField("col1", XXXType())
              ,StructField("col2",YYYType())
             ]
           )
        ,input_types=[IntegerType()]
        ,name = "MY_UDTF_NAME"
        ,is_permanent = False
        ,stage_location = "@some_stage" # Mandatory if is_permanent is True
        ,replace = True                 # Default is False
        ,session = mySessionObject
        # For arguments named imports and packages see earlier in this lesson
    )
)

# Use the function
from snowflake.snowpark.functions import call_table_function
(session.table_function(   # Produces a DataFrame
     call_table_function(  # Produces a TableFunctionCall object
          "MY_UDTF_NAME"
         ,<some Column as arg1>
         ,<some Column as arg12>
         , ...
       )
    .show()
  )
)
```

> **&#128221; Note:** See documentation for further details:
> - [Creating User-Defined Table Functions (UDTFs) for DataFrames in Python](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-udtfs.html#creating-user-defined-table-functions-udtfs-for-dataframes-in-python)
> - [snowflake.snowpark.table_function.TableFunctionCall(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.table_function.TableFunctionCall.html#snowflake-snowpark-table-function-tablefunctioncall)
> - [snowflake.snowpark.functions.call_table_function(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.call_table_function.html#snowflake.snowpark.functions.call_table_function)

---

We start by developing the UDTF handler class. 

In [16]:
# Import the types and objects needed
from snowflake.snowpark.types import StructType, StructField 
import math

#   Table function returning rows of results from methods
#   in math such as sqrt(...), sin(...), and log(...).
#   This UDTF takes one argument of type float. 
class NumberStuff:
    
    # Does nothing. Included here for verbosity. 
    def __init__(self):
        return None
    
    # Invoked with each input row
    def process(self, num):
        typ = str(type(num)).split("'")[1]
        list_of_tuples = (
           [
               # col1 operation       col2 results
             (f"Number ({typ})",      num)
            ,(f"math.sqrt({num})",    math.sqrt(num))
            ,(f"math.pow({num},2)",   math.pow(num,2))
            ,(f"math.pow({num},3)",   math.pow(num,3))
            ,(f"math.log({num})",     math.log(num))
            ,(f"math.log10({num})",   math.log10(num))
            ,(f"math.sin({num})",     math.sin(num))
            ,(f"math.cos({num})",     math.cos(num))
            ,(f"math.tan({num})",     math.tan(num))
           ]
         )
        return list_of_tuples
    
    # Invoked once after all input values for this partition have processed.
    # Does nothing. Included here for verbosity. 
    def end_partition(self):
        return None
    
print(f"Class NumberStuff created: {NumberStuff}")    

Class NumberStuff created: <class '__main__.NumberStuff'>


<a id="It_s_Just_a_Class_It_s_Just_Methods"></a>
### 5D. It's Just a Class - It's Just Methods

The handler class contains methods that make it compatible with Snowpark through lifecycle or "callback" methods. However, its logic can function independently of Snowpark transformations. Since the class outputs tuples, there's no requirement for the Snowpark library to be present.

In [17]:
# Create an instance
my_number_stuff = NumberStuff()

# Invoke the process method
row_list = my_number_stuff.process(25.0)

# Iterate through the returned rows
print("Each Row object from calling process(25)")
for row in row_list:
    print(row)

Each Row object from calling process(25)
('Number (float)', 25.0)
('math.sqrt(25.0)', 5.0)
('math.pow(25.0,2)', 625.0)
('math.pow(25.0,3)', 15625.0)
('math.log(25.0)', 3.2188758248682006)
('math.log10(25.0)', 1.3979400086720377)
('math.sin(25.0)', -0.13235175009777303)
('math.cos(25.0)', 0.9912028118634736)
('math.tan(25.0)', -0.13352640702153587)


<a id="Registering_a_Python_UDTF"></a>
### 5E. Registering a Python UDTF

We register the temporary UDTF much like we register a temporary scalar UDF. 

See the above example of registering a Python scalar UDF for the argument details. The biggest difference is with the argument `output_schema`.  With a scalar UDF, you indicate a `return_type` such as `StringType()` or `IntegerType()`. UDTFs produce rows of data. We specify this with a list of `StructField(...)` instances. Additionally, a single function does not power a UDTF like it does with a UDF (`func_name`). For a UDTF, the `handler` class must be passed when registering. 

In [18]:
# We will need a few imports
from snowflake.snowpark.functions import udtf
from snowflake.snowpark.types import StructType, StructField, StringType, DoubleType

# Create an instance of UserDefinedTableFunction
my_udtf_instance = (
    udtf(
         handler = NumberStuff        # Class that will power this UDTF. Notice no double-quote marks. 
        ,output_schema = StructType(  # Rows are returned. What are the columns? 
            [  StructField("MATH_OPERATION", StringType())  # Results column 1 type
              ,StructField("RESULTS", DoubleType())         # Results column 2 type
            ]
           )
        ,input_types=[DoubleType()]   # A single double is passed. (Cute, yes? See it? Get it?)
        ,name = "NUM_STUFF"           # Name used in SQL text
        ,is_permanent = False         # Temp UDTF
        ,replace = True               # In case we want to run this cell again
        ,session = session            # Session to use to register this UDTF
        # For the arguments named imports and packages, see the docs
    )
)

print(f"UDTF my_udtf_instance created: {my_udtf_instance}")

UDTF my_udtf_instance created: <snowflake.snowpark.udtf.UserDefinedTableFunction object at 0x7f4c0067d280>


<a id="Invoking_Python_UDTFs"></a>
### 5F. Invoking Python UDTFs

We can now invoke our UDTF using `Session.table_function(...)`. This function returns a `DataFrame` object.  

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Session.table_function(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.table_function.html#snowflake.snowpark.Session.table_function)

In [19]:
# Imports
from snowflake.snowpark.functions import call_table_function, lit

# Invoke with the function's name using call_table_function
print("Invoking the NUM_STUFF table function using session.table_function(...) and passing the results of call_table_function(...)")
(session.table_function(
     call_table_function(  # Produces a DataFrame
          "NUM_STUFF"
         ,lit(25.0)
      )
   )
  .show()
)

Invoking the NUM_STUFF table function using session.table_function(...) and passing the results of call_table_function(...)
-------------------------------------------
|"MATH_OPERATION"  |"RESULTS"             |
-------------------------------------------
|Number (float)    |25.0                  |
|math.sqrt(25.0)   |5.0                   |
|math.pow(25.0,2)  |625.0                 |
|math.pow(25.0,3)  |15625.0               |
|math.log(25.0)    |3.2188758248682006    |
|math.log10(25.0)  |1.3979400086720377    |
|math.sin(25.0)    |-0.13235175009777303  |
|math.cos(25.0)    |0.9912028118634736    |
|math.tan(25.0)    |-0.13352640702153587  |
-------------------------------------------



In [20]:
# Invoke using our instance of UserDefinedTableFunction
print("Invoking the NUM_STUFF table function using session.table_function(...) and passing an instance of TableFunctionCall")
(session.table_function(
    my_udtf_instance(  # Produces a TableFunctionCall instance
        lit(25.0)
     )
   )
  .show()
)

Invoking the NUM_STUFF table function using session.table_function(...) and passing an instance of TableFunctionCall
-------------------------------------------
|"MATH_OPERATION"  |"RESULTS"             |
-------------------------------------------
|Number (float)    |25.0                  |
|math.sqrt(25.0)   |5.0                   |
|math.pow(25.0,2)  |625.0                 |
|math.pow(25.0,3)  |15625.0               |
|math.log(25.0)    |3.2188758248682006    |
|math.log10(25.0)  |1.3979400086720377    |
|math.sin(25.0)    |-0.13235175009777303  |
|math.cos(25.0)    |0.9912028118634736    |
|math.tan(25.0)    |-0.13352640702153587  |
-------------------------------------------



<a id="Using_a_Python_UDTF_in_SQL"></a>
### 5G. Using a Python UDTF in SQL

We can use the registered UDTF in a SQL statement. 

```sql
SELECT * FROM TABLE(myTableFunction(<params>));
```

In [21]:
# Use our UDTF in a SQL statement
print("""Table function results from SQL invocation:
            SELECT * FROM TABLE(NUM_STUFF(25::DOUBLE))"""
       )
(session.sql("SELECT * FROM TABLE(NUM_STUFF(25::DOUBLE))")
    .show(15)
)

Table function results from SQL invocation:
            SELECT * FROM TABLE(NUM_STUFF(25::DOUBLE))
-------------------------------------------
|"MATH_OPERATION"  |"RESULTS"             |
-------------------------------------------
|Number (float)    |25.0                  |
|math.sqrt(25.0)   |5.0                   |
|math.pow(25.0,2)  |625.0                 |
|math.pow(25.0,3)  |15625.0               |
|math.log(25.0)    |3.2188758248682006    |
|math.log10(25.0)  |1.3979400086720377    |
|math.sin(25.0)    |-0.13235175009777303  |
|math.cos(25.0)    |0.9912028118634736    |
|math.tan(25.0)    |-0.13352640702153587  |
-------------------------------------------



---
<a id="AE6_cleanup"></a>

## 6. Clean Up and Close Session

Best practice is to clean up demo objects, suspend our warehouse, and close the Snowpark Session object.

In [22]:
close_session_and_clean_up(get_lesson())

Lesson object (lesson) found.
Dropping schema RAVEN_DB.UDFS_AND_UDTFS_PY_LESSON
--------------------------------------------------
|"status"                                        |
--------------------------------------------------
|UDFS_AND_UDTFS_PY_LESSON successfully dropped.  |
--------------------------------------------------

Suspending warehouse RAVEN_WH
------------------------------------
|"status"                          |
------------------------------------
|Statement executed successfully.  |
------------------------------------

Closing session
Session closed


### &#10071; `Shut Down Kernel`
> After completing the activities in a notebook and before moving on to the next exercise, shut down the completed notebook by right-clicking on the notebook name and selecting `Shut Down Kernel`.